# LAB 07 — TRANSFORMER COMPONENTS

## 1. Mục tiêu

Tìm hiểu và triển khai các thành phần quan trọng bên trong Transformer:

1. Feed Forward Network (FFN).
2. Residual Connection.
3. Layer Normalization.

Phần này sẽ tính toán thủ công, xây dựng các thành phần bằng PyTorch và kiểm tra kết quả thực nghiệm.

## 2. Feed Forward Network (FFN)

### 2.1. Kiến thức nền tảng

Feed Forward Network là mạng truyền thẳng được áp dụng độc lập lên từng token sau quá trình Self-Attention.

Một FFN cơ bản gồm hai lớp Linear và một hàm kích hoạt ReLU.

Công thức:

FFN(x) = W2 * ReLU(W1 * x + b1) + b2

Trong đó:

- \(x\): Vector đầu vào.
- \(W1, W2\): Ma trận trọng số.
- \(b1, b2\): Vector bias.
- ReLU: Hàm kích hoạt phi tuyến.

In [2]:
import numpy as np
import torch
import torch.nn as nn

print("PyTorch:", torch.__version__)

PyTorch: 2.14.1+cpu


In [3]:
x = torch.tensor([1.0, 2.0])

W1 = torch.eye(2)
b1 = torch.tensor([0.0, -1.0])

h = torch.matmul(W1, x) + b1
h_relu = torch.relu(h)

print("Input:", x)
print("After Linear:", h)
print("After ReLU:", h_relu)

Input: tensor([1., 2.])
After Linear: tensor([1., 1.])
After ReLU: tensor([1., 1.])


In [4]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)

        return x

In [5]:
d_model = 128
d_ff = 512

batch_size = 8
sequence_length = 100

x = torch.randn(batch_size, sequence_length, d_model)

ffn = FeedForwardNetwork(d_model, d_ff)

output = ffn(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Number of parameters:", sum(p.numel() for p in ffn.parameters()))

Input shape: torch.Size([8, 100, 128])
Output shape: torch.Size([8, 100, 128])
Number of parameters: 131712


### Nhận xét kết quả (Observation)

- Kích thước đầu vào: (8, 100, 128)
- Kích thước đầu ra: (8, 100, 128)
- Tổng số tham số: 131.712

Mạng FFN mở rộng vector biểu diễn của mỗi token từ 128 chiều lên 512 chiều, sau đó áp dụng hàm kích hoạt ReLU và đưa vector trở về 128 chiều.

Kích thước tensor đầu vào và đầu ra không thay đổi. Điều này cho thấy FFN có thể xử lý thông tin của từng token mà vẫn giữ nguyên cấu trúc dữ liệu ban đầu.

## Residual Connection (Kết nối phần dư)

Residual Connection là cơ chế cộng trực tiếp đầu vào ban đầu với đầu ra của một lớp xử lý.

Công thức:

y = x + F(x)

Trong đó:
- x: dữ liệu đầu vào.
- F(x): kết quả sau khi đi qua lớp xử lý.
- y: kết quả sau phép cộng.

**Mục đích:**
- Giữ lại thông tin từ đầu vào ban đầu.
- Hỗ trợ quá trình truyền thông tin qua nhiều lớp.
- Giúp hạn chế vấn đề mất dần gradient khi huấn luyện mạng sâu.

**Lưu ý:** Hai tensor x và F(x) phải có kích thước tương thích để thực hiện phép cộng.

### Prediction

**Dự đoán 1:**
Nếu đầu vào x = [1, 2, 3] và đầu ra lớp xử lý F(x) = [0.5, 1, 1.5], kết quả sau Residual Connection là [1.5, 3, 4.5].

**Dự đoán 2:**
Nếu bỏ phép cộng x + F(x), kết quả chỉ còn F(x). Thông tin đầu vào ban đầu không còn được giữ trực tiếp thông qua kết nối phần dư.

In [6]:
x = torch.tensor([1.0, 2.0, 3.0])
fx = torch.tensor([0.5, 1.0, 1.5])

output = x + fx

print("Input:", x)
print("Layer output:", fx)
print("Residual output:", output)

Input: tensor([1., 2., 3.])
Layer output: tensor([0.5000, 1.0000, 1.5000])
Residual output: tensor([1.5000, 3.0000, 4.5000])


### Nhận xét kết quả (Observation)

- Đầu vào: [1, 2, 3]
- Đầu ra lớp xử lý: [0.5, 1, 1.5]
- Đầu ra sau kết nối phần dư: [1.5, 3, 4.5]

Kết quả thực nghiệm khớp với dự đoán.

Kết nối phần dư thực hiện phép cộng giữa đầu vào ban đầu và đầu ra của lớp xử lý. Nhờ đó, thông tin từ đầu vào được giữ lại trong kết quả, đồng thời kết hợp với thông tin mới được xử lý.

In [7]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
fx = x * 2

output = x + fx
loss = output.sum()

loss.backward()

print("Output:", output)
print("Gradient:", x.grad)

Output: tensor([3., 6., 9.], grad_fn=<AddBackward0>)
Gradient: tensor([3., 3., 3.])


### Nhận xét kết quả Gradient

- Đầu ra: [3, 6, 9]
- Gradient: [3, 3, 3]

Kết quả thực nghiệm khớp với dự đoán.

Gradient được truyền qua cả nhánh xử lý và nhánh kết nối trực tiếp. Nhờ đó, kết nối phần dư tạo thêm một đường truyền gradient, hỗ trợ quá trình huấn luyện mạng nơ-ron sâu và hạn chế hiện tượng mất dần gradient.

## Layer Normalization (Chuẩn hóa lớp)

Layer Normalization là kỹ thuật chuẩn hóa dữ liệu dựa trên các đặc trưng của từng mẫu dữ liệu.

Mục đích:
- Đưa dữ liệu về trạng thái có phân phối ổn định hơn.
- Giúp quá trình huấn luyện mô hình ổn định.
- Hỗ trợ mô hình hội tụ hiệu quả hơn.

Đối với Transformer, Layer Normalization thường được áp dụng trên chiều đặc trưng của từng token.

### Prediction

Với vector đầu vào [1, 2, 3, 4]:

1. Sau khi chuẩn hóa, giá trị trung bình được dự đoán bằng 0.
2. Độ lệch chuẩn được dự đoán bằng 1.

Layer Normalization giúp các giá trị trong vector có phân phối ổn định hơn.

In [8]:
x = torch.tensor([1.0, 2.0, 3.0, 4.0])

layer_norm = nn.LayerNorm(4)
output = layer_norm(x)

print("Input:", x)
print("Output:", output)
print("Mean:", output.mean().item())
print("Standard deviation:", output.std(unbiased=False).item())

Input: tensor([1., 2., 3., 4.])
Output: tensor([-1.3416, -0.4472,  0.4472,  1.3416],
       grad_fn=<NativeLayerNormBackward0>)
Mean: 0.0
Standard deviation: 0.9999960064888


### Nhận xét kết quả (Observation)

- Vector đầu vào: [1, 2, 3, 4]
- Vector sau chuẩn hóa: [-1.3416, -0.4472, 0.4472, 1.3416]
- Giá trị trung bình: 0.0
- Độ lệch chuẩn: 0.999996

Kết quả thực nghiệm khớp với dự đoán.

Layer Normalization đưa các giá trị trong vector về trạng thái có giá trị trung bình gần 0 và độ lệch chuẩn gần 1. Sai số nhỏ xuất hiện do tham số epsilon được sử dụng để đảm bảo tính ổn định trong quá trình tính toán.

In [9]:
x = torch.randn(2, 3, 4)

layer_norm = nn.LayerNorm(4)
output = layer_norm(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Mean along last dimension:")
print(output.mean(dim=-1))
print("Standard deviation along last dimension:")
print(output.std(dim=-1, unbiased=False))

Input shape: torch.Size([2, 3, 4])
Output shape: torch.Size([2, 3, 4])
Mean along last dimension:
tensor([[ 1.4901e-08,  0.0000e+00,  1.4901e-07],
        [ 7.4506e-09,  0.0000e+00, -5.9605e-08]], grad_fn=<MeanBackward1>)
Standard deviation along last dimension:
tensor([[1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000]], grad_fn=<StdBackward0>)


### Nhận xét kết quả (Observation)

- Kích thước đầu vào: (2, 3, 4)
- Kích thước đầu ra: (2, 3, 4)
- Giá trị trung bình của mỗi token: gần bằng 0.
- Độ lệch chuẩn của mỗi token: gần bằng 1.

Kết quả thực nghiệm khớp với dự đoán.

Layer Normalization thực hiện chuẩn hóa độc lập trên chiều đặc trưng của từng token. Vì vậy, mỗi token có giá trị trung bình gần 0 và độ lệch chuẩn gần 1, trong khi kích thước tensor được giữ nguyên.

Các sai số rất nhỏ trong giá trị trung bình xuất hiện do giới hạn độ chính xác của phép tính số thực.